# TRIAGE-EG D1 — Grounding / Candidate Error Attribution Audit

Diagnostic-only run on `DEV_CROSS_60`. It reproduces and hashes frozen `G1_COVERAGE_COARSE` before loading GT, preserves full Stage1 score vectors only in memory, audits BTC/CLIP/T3/global/G1/TRAKE failure stages, and creates blinded translation QC. It does not run L21, SEALED, M1, M2, M3, VLM, Agent, Event Graph, model download, or parameter tuning.

Required Kaggle inputs: raw AIC dataset, finalized TEAM-EVAL dev v1, Stage1 exact index, Stage1B verified contract, Stage1E language contract, OpenAI CLIP ViT-B/32 offline asset, and OPUS-MT vi-en offline asset. The historical E2E-G1 v0.1 bundle is optional. Internet is used only to clone `TRIAGEEG` when the repo is absent; runtime model download is disabled. Download output: `/kaggle/working/triage_eg_d1_v01_bundle.zip`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from time import monotonic
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_EVAL_INPUT = Path(
    os.environ.get(
        "AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-team-eval-dev-v1"
    )
)
STAGE1_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1E_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    )
)
CLIP_INPUT = Path(
    os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
)
OPUS_INPUT = Path(
    os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
)
HISTORICAL_INPUT = Path(
    os.environ.get(
        "AIC_HISTORICAL_E2EG1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-e2eg1-v01-bundle"
    )
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_d1_v01")
RUNTIME_ROOT = Path("/kaggle/working/triage_eg_d1_stage2_runtime")
ZIP_PATH = Path("/kaggle/working/triage_eg_d1_v01_bundle.zip")
TEAM_EVAL_REPACKED_ZIP = Path("/kaggle/working/aic2026_team_eval_dev_v1_repacked.zip")
EXTRACT_ROOT = Path("/kaggle/working/aic2026_team_eval_dev_v1_extracted")
INFERENCE_ROOT = Path("/kaggle/working/triage_eg_d1_inference_only")
TEMP_PREDICTION_ROOT = Path("/kaggle/working/triage_eg_d1_prediction_temp")
MATERIALIZED_ROOTS = {
    name: Path("/kaggle/working") / path
    for name, path in {
        "stage1": "triage_eg_stage1_materialized",
        "stage1b": "triage_eg_stage1b_materialized",
        "stage1e": "triage_eg_stage1e_materialized",
        "clip": "aic2026_openai_clip_materialized",
        "opus": "aic2026_opus_materialized",
    }.items()
}
for target in (
    OUTPUT_ROOT,
    RUNTIME_ROOT,
    EXTRACT_ROOT,
    INFERENCE_ROOT,
    TEMP_PREDICTION_ROOT,
    *MATERIALIZED_ROOTS.values(),
):
    if target.exists():
        if target.parent != Path("/kaggle/working"):
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {target}")
        shutil.rmtree(target)
for target in (ZIP_PATH, TEAM_EVAL_REPACKED_ZIP):
    target.unlink(missing_ok=True)
print(
    {
        "required_inputs": {
            "raw_dataset": str(DATA_INPUT),
            "team_eval_dev_bundle": str(TEAM_EVAL_INPUT),
            "stage1_exact_index": str(STAGE1_INPUT),
            "stage1b_verified_contract": str(STAGE1B_INPUT),
            "stage1e_language_contract": str(STAGE1E_INPUT),
            "openai_clip_offline_asset": str(CLIP_INPUT),
            "opus_mt_vi_en_offline_asset": str(OPUS_INPUT),
        },
        "optional_input": {"historical_e2eg1_bundle": str(HISTORICAL_INPUT)},
        "internet_required": "ONLY_FOR_GIT_CLONE_IF_REPO_NOT_PRESENT",
        "model_download_required": False,
        "primary_benchmark": "DEV_CROSS_60_ONLY",
        "output_zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH, MAX_DIRECTORIES = 7, 10000
TEAM_EVAL_REQUIRED_MEMBERS = (
    "README.md",
    "benchmark_registry.json",
    "benchmarks/dev_cross_60/annotation_audit.jsonl",
    "benchmarks/dev_cross_60/gt.jsonl",
    "benchmarks/dev_cross_60/manifest.json",
    "benchmarks/dev_cross_60/queries.jsonl",
    "benchmarks/dev_l21_150/annotation_audit.jsonl",
    "benchmarks/dev_l21_150/gt.jsonl",
    "benchmarks/dev_l21_150/manifest.json",
    "benchmarks/dev_l21_150/queries.jsonl",
)


def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError("Kaggle input discovery exceeded bound")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )


def resolve_file(hint, filename, *, optional=False):
    hint = Path(hint)
    if hint.is_file() and hint.name == filename:
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory / filename
            for directory in bounded_dirs(root)
            if (directory / filename).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {filename}; found {matches}")
    return matches[0]


def resolve_root(hint, marker, *, optional=False):
    hint, marker = Path(hint), Path(marker)
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if (directory / marker).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one root with {marker}; found {matches}")
    return matches[0]


def is_team_eval_root(root):
    return Path(root).is_dir() and all(
        (Path(root) / member).is_file() for member in TEAM_EVAL_REQUIRED_MEMBERS
    )


def resolve_team_eval_root(hint, *, optional=False):
    hint = Path(hint)
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if is_team_eval_root(directory)
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one extracted TEAM-EVAL root; found {matches}")
    return matches[0]


def resolve_dataset(hint):
    hint, marker = Path(hint), Path("map-keyframes-aic25-b1/map-keyframes")
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory
            for directory in bounded_dirs(root)
            if (directory / marker).is_dir() and any(directory.glob("Videos_*/video"))
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root; found {matches}")
    return matches[0]


DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_MOUNTED_ZIP = resolve_file(TEAM_EVAL_INPUT, "aic2026_team_eval_dev_v1.zip", optional=True)
TEAM_EVAL_MOUNTED_ROOT = resolve_team_eval_root(TEAM_EVAL_INPUT, optional=True)
HISTORICAL_E2EG1_ZIP = resolve_file(
    HISTORICAL_INPUT, "triage_eg_e2eg1_v01_bundle.zip", optional=True
)
HISTORICAL_E2EG1_ROOT = (
    None
    if HISTORICAL_E2EG1_ZIP
    else resolve_root(HISTORICAL_INPUT, "predictions/dev_cross_60_g1.jsonl", optional=True)
)
HISTORICAL_E2EG1_SOURCE = HISTORICAL_E2EG1_ZIP or HISTORICAL_E2EG1_ROOT
print(
    {
        "raw_dataset": str(DATASET_ROOT),
        "team_eval_zip": str(TEAM_EVAL_MOUNTED_ZIP) if TEAM_EVAL_MOUNTED_ZIP else None,
        "team_eval_extracted_root": str(TEAM_EVAL_MOUNTED_ROOT) if TEAM_EVAL_MOUNTED_ROOT else None,
        "historical_e2eg1_optional": str(HISTORICAL_E2EG1_SOURCE)
        if HISTORICAL_E2EG1_SOURCE
        else "NOT_MOUNTED_OPTIONAL",
    }
)

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
if not (REPO_DIR / "src/triage_eg/diagnostics/d1_grounding_attribution/runner.py").is_file():
    raise RuntimeError(
        "TRIAGEEG ref does not contain D1; publish reviewed source changes before Kaggle execution"
    )
sys.path.insert(0, str(REPO_DIR / "src"))
HEAD = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
GIT_STATUS = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
if BRANCH != REPO_REF:
    raise RuntimeError(f"Expected branch {REPO_REF}, resolved {BRANCH}")
print({"branch": BRANCH, "HEAD": HEAD, "git_status": GIT_STATUS or "CLEAN"})

In [ ]:
import yaml

from aic2026_eval.io import sha256_file
from triage_eg.diagnostics.d1_grounding_attribution import D1Settings
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root


def repack_team_eval_root(source_root, destination):
    source_root, destination = Path(source_root), Path(destination)
    missing = [
        member for member in TEAM_EVAL_REQUIRED_MEMBERS if not (source_root / member).is_file()
    ]
    if missing:
        raise RuntimeError(f"Extracted TEAM-EVAL bundle incomplete: {missing}")
    destination.unlink(missing_ok=True)
    with ZipFile(destination, "w", compression=ZIP_DEFLATED) as archive:
        for member in TEAM_EVAL_REQUIRED_MEMBERS:
            if "sealed" in member.casefold():
                raise RuntimeError("SEALED_CONTENT_REJECTED")
            info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, (source_root / member).read_bytes())
    return destination.resolve(strict=True)


if TEAM_EVAL_MOUNTED_ZIP:
    TEAM_EVAL_ZIP, TEAM_EVAL_SOURCE = TEAM_EVAL_MOUNTED_ZIP, "KAGGLE_ZIP"
elif TEAM_EVAL_MOUNTED_ROOT:
    TEAM_EVAL_ZIP = repack_team_eval_root(TEAM_EVAL_MOUNTED_ROOT, TEAM_EVAL_REPACKED_ZIP)
    TEAM_EVAL_SOURCE = "KAGGLE_EXTRACTED_BUNDLE_REPACKED"
else:
    raise RuntimeError("Missing finalized TEAM-EVAL development ZIP or extracted bundle root")


def optional_search_root(hint):
    return None if Path(hint).exists() else SEARCH_ROOT


STAGE1_ROOT = resolve_stage1_root(
    STAGE1_INPUT,
    search_root=optional_search_root(STAGE1_INPUT),
    materialize_root=MATERIALIZED_ROOTS["stage1"],
)
STAGE1B_ROOT, _ = resolve_input_root(
    STAGE1B_INPUT,
    required=(
        "stage1b_summary.json",
        "encoder/selected_encoder_contract.json",
        "encoder/runtime_adapter_manifest.json",
    ),
    materialize_root=MATERIALIZED_ROOTS["stage1b"],
    search_root=optional_search_root(STAGE1B_INPUT),
    archive_keyword="stage1b",
)
STAGE1E_ROOT, _ = resolve_input_root(
    STAGE1E_INPUT,
    required=("stage1e_summary.json", "language_path_contract.json"),
    materialize_root=MATERIALIZED_ROOTS["stage1e"],
    search_root=optional_search_root(STAGE1E_INPUT),
    archive_keyword="stage1e",
)
CLIP_ROOT, _ = resolve_input_root(
    CLIP_INPUT,
    required=("checkpoint/ViT-B-32.pt", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED_ROOTS["clip"],
    search_root=optional_search_root(CLIP_INPUT),
    archive_keyword="clip",
)
OPUS_ROOT, _ = resolve_input_root(
    OPUS_INPUT,
    required=("model/config.json", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED_ROOTS["opus"],
    search_root=optional_search_root(OPUS_INPUT),
    archive_keyword="opus",
)
SETTINGS = D1Settings()
D1_CONFIG = yaml.safe_load(
    (REPO_DIR / "configs/experiments/triage_d1_grounding_attribution_v01.yaml").read_text(
        encoding="utf-8"
    )
)
CONFIG_SETTINGS = {key: D1_CONFIG[key] for key in SETTINGS.as_dict()}
if SETTINGS.as_dict() != CONFIG_SETTINGS or D1_CONFIG.get("parameter_sweep") is not False:
    raise RuntimeError("Frozen D1 YAML/dataclass contract mismatch")
EXPECTED_TEAM_EVAL_SHA256 = "0d455e07bc866c49d2f60af3d0ffeff7c953e3218530d254a74f8f52abd17b07"
if sha256_file(TEAM_EVAL_ZIP) != EXPECTED_TEAM_EVAL_SHA256:
    raise RuntimeError("TEAM-EVAL development bundle SHA-256 mismatch")
print(
    {
        "team_eval": str(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
        "team_eval_sha256": sha256_file(TEAM_EVAL_ZIP),
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
        "historical_e2eg1_optional": str(HISTORICAL_E2EG1_SOURCE)
        if HISTORICAL_E2EG1_SOURCE
        else None,
    }
)
print("FROZEN_D1_CONFIG=", json.dumps(SETTINGS.as_dict(), indent=2))

In [ ]:
from triage_eg.e2eg1 import extract_development_bundle

TEAM_EVAL_ROOT = extract_development_bundle(TEAM_EVAL_ZIP, EXTRACT_ROOT)
with ZipFile(TEAM_EVAL_ZIP) as archive:
    MEMBERS = archive.namelist()
assert not any("sealed" in name.casefold() for name in MEMBERS)
CROSS_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_cross_60"
if not all(
    (CROSS_ROOT / name).is_file() for name in ("queries.jsonl", "gt.jsonl", "manifest.json")
):
    raise RuntimeError("DEV_CROSS_60 is incomplete")
print({"cross": str(CROSS_ROOT), "DEV_L21_150_RUN": False, "SEALED_ACCESS_GATE": "PASS"})

In [ ]:
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage2 import config_from_yaml

STARTUP_STARTED = monotonic()
STAGE2_CONFIG = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime_gpu.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=RUNTIME_ROOT,
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=HEAD,
)
PIPELINE = SafeCoveragePipeline.load_once(STAGE2_CONFIG, DATASET_ROOT)
STARTUP_SECONDS = monotonic() - STARTUP_STARTED
print(
    {
        "startup_seconds": STARTUP_SECONDS,
        "resources": PIPELINE.runtime.runtime_manifest(),
        "M1": False,
        "M2": False,
        "M3": False,
        "EVENT_GRAPH": False,
        "VLM": False,
        "AGENT": False,
    }
)

In [ ]:
from triage_eg.e2eg1 import materialize_inference_only

CROSS_INFERENCE = materialize_inference_only(CROSS_ROOT, INFERENCE_ROOT / "dev_cross_60")
assert {path.name for path in CROSS_INFERENCE.iterdir()} == {"queries.jsonl"}
print(
    {
        "cross_inference_only": str(CROSS_INFERENCE),
        "members": ["queries.jsonl"],
        "GT_AVAILABLE_TO_INFERENCE": False,
    }
)

In [ ]:
from triage_eg.diagnostics.d1_grounding_attribution import (
    capture_inference_snapshot,
    run_g1_reproduction,
    verify_historical_reproduction,
    write_blind_translation_artifacts,
)

D1_RUN = run_g1_reproduction(PIPELINE, CROSS_INFERENCE, OUTPUT_ROOT, TEMP_PREDICTION_ROOT)
SNAPSHOT = capture_inference_snapshot(PIPELINE, D1_RUN)
BLIND_ROWS, TRANSLATION_SUMMARY = write_blind_translation_artifacts(SNAPSHOT, OUTPUT_ROOT)
REPRODUCTION = verify_historical_reproduction(D1_RUN, HISTORICAL_E2EG1_SOURCE, OUTPUT_ROOT)
print(
    {
        "variant": D1_RUN["variant"],
        "prediction_sha256": D1_RUN["sha256"],
        "validation": D1_RUN["validation"],
        "G1_REPRODUCTION": REPRODUCTION["status"],
        "translation_units": len(BLIND_ROWS),
        "score_vectors_in_memory_only": len(SNAPSHOT.units),
        "GT_LEAKAGE_GATE": "PASS",
    }
)

In [ ]:
from aic2026_eval.io import read_jsonl
from triage_eg.diagnostics.d1_grounding_attribution import run_post_gt_attribution

# First GT read: hash, strict validation, snapshot, and optional reproduction passed.
CROSS_GT = read_jsonl(CROSS_ROOT / "gt.jsonl")
AUDIT = run_post_gt_attribution(
    SNAPSHOT,
    D1_RUN,
    CROSS_GT,
    PIPELINE,
    OUTPUT_ROOT,
    settings=SETTINGS,
    translation_summary=TRANSLATION_SUMMARY,
)
print(
    json.dumps(
        {
            "single_event_summary": AUDIT["single_summary"],
            "trake_summary": AUDIT["trake_summary"],
            "attribution_summary": AUDIT["attribution_summary"],
        },
        indent=2,
    )
)

In [ ]:
from IPython.display import Image, display

from triage_eg.diagnostics.d1_grounding_attribution import render_review_sheets

REVIEW_PATHS = render_review_sheets(PIPELINE, AUDIT, BLIND_ROWS, OUTPUT_ROOT)
assert (
    len([path for path in REVIEW_PATHS if path.parent.name == "review"])
    <= SETTINGS.max_review_cases
)
for path in [value for value in REVIEW_PATHS if value.parent.name == "montages"]:
    display(Image(filename=str(path)))
print(
    {
        "review_artifacts": len(REVIEW_PATHS),
        "MAX_REVIEW_CASES": SETTINGS.max_review_cases,
        "GT_OVERLAY_PHASE": "POST_INFERENCE_EVALUATION_ONLY",
    }
)

In [ ]:
from triage_eg.diagnostics.d1_grounding_attribution import create_bundle, write_manifests

write_manifests(
    OUTPUT_ROOT,
    pipeline=PIPELINE,
    settings=SETTINGS,
    dataset_root=DATASET_ROOT,
    team_eval_bundle=TEAM_EVAL_ZIP,
    historical_e2eg1_source=HISTORICAL_E2EG1_SOURCE,
    branch=BRANCH,
    git_commit=HEAD,
    reproduction=REPRODUCTION,
    run=D1_RUN,
)
if TEMP_PREDICTION_ROOT.exists():
    shutil.rmtree(TEMP_PREDICTION_ROOT)
BUNDLE = create_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(BUNDLE) as archive:
    BUNDLE_MEMBERS = archive.namelist()
assert not any(
    name.casefold().endswith((".mp4", ".npy", ".npz", ".pt", ".pth", ".bin"))
    for name in BUNDLE_MEMBERS
)
assert not any("sealed" in name.casefold() for name in BUNDLE_MEMBERS)
assert not any(
    name.endswith(("gt.jsonl", "queries.jsonl", "annotation_audit.jsonl"))
    for name in BUNDLE_MEMBERS
)
print(
    {
        "download_zip": str(BUNDLE),
        "size_bytes": BUNDLE.stat().st_size,
        "members": len(BUNDLE_MEMBERS),
    }
)

In [ ]:
from triage_eg.diagnostics.d1_grounding_attribution import formal_report_lines

for line in formal_report_lines(
    git_commit=HEAD,
    reproduction=REPRODUCTION,
    translation_summary=TRANSLATION_SUMMARY,
    audit=AUDIT,
    zip_path=BUNDLE,
):
    print(line)
print(
    "INPUTS_USED=",
    {
        "raw_dataset": str(DATASET_ROOT),
        "team_eval_dev_zip": str(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
        "historical_e2eg1_optional": str(HISTORICAL_E2EG1_SOURCE)
        if HISTORICAL_E2EG1_SOURCE
        else None,
    },
)
PIPELINE.close()